In [ ]:
# import os
# os.makedirs("./AI_Cache/huggingface", exist_ok=True)
# os.makedirs("./AI_Cache/torch", exist_ok=True)
# os.environ["HF_HOME"] = r"./AI_Cache/huggingface"
# os.environ["TORCH_HOME"] = r"./AI_Cache/torch"
print("running")
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
from pathlib import Path
from typing import Tuple, Dict, Optional, List, Literal, Sequence, Union
import warnings
from PIL import Image
import clip
import ast
import datasets
import random

warnings.filterwarnings("ignore")
torch.multiprocessing.set_sharing_strategy('file_system')

def set_seed(seed=114514):
    random.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:


# ====================== 从 EEG Project Sample Code 导入的数据加载函数 ======================
def _selected_channel_indices_from_jsonl(
    selected_channels: Union[str, Sequence[str]],
    eeg_channel_jsonl: Union[str, Path],
) -> List[int]:
    """Map EEG channel names to channel indices."""
    if isinstance(selected_channels, str):
        selected_channels = [selected_channels]
    selected_channels = list(selected_channels)

    channel_names: List[str] = []
    with open(eeg_channel_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            name = item.get("name") or item.get("channel_name") or item.get("label")
            if name is None:
                raise KeyError(
                    "Each JSONL record must contain one of: 'name', 'channel_name', or 'label'."
                )
            channel_names.append(str(name))

    name_to_index = {name: idx for idx, name in enumerate(channel_names)}

    missing = [ch for ch in selected_channels if ch not in name_to_index]
    if missing:
        raise ValueError(f"Unknown EEG channels: {missing}")

    return [name_to_index[ch] for ch in selected_channels]


def load_eeg_dataset(
    *,
    data_directory: Union[str, Path],
    split: Literal["train", "test"],
    avg_trials: bool = True,
    selected_channels: Optional[Union[str, Sequence[str]]] = None,
    eeg_channel_jsonl: Union[str, Path] = "image-eeg-data/EEG_CHANNELS.jsonl",
) -> datasets.Dataset:
    """Build a Hugging Face dataset for the released EEG data.
    
    Returns dataset with columns:
    - `eeg`: Array2D float32 [C, T]
    - `image_id`: string
    """
    pt_path = Path(data_directory).joinpath(f"{split}.pt")
    loaded = torch.load(str(pt_path), weights_only=False)

    x = torch.as_tensor(loaded["eeg"])  # [N, TRIAL, C, T] or [N, C, T]
    if x.ndim == 4:
        if avg_trials:
            x = x.mean(dim=1)  # [N, C, T]
        else:
            x = x.reshape(-1, *x.shape[2:])  # [N * TRIAL, C, T]
    elif x.ndim != 3:
        raise ValueError(f"Unexpected EEG shape: {tuple(x.shape)} in {pt_path}")

    if selected_channels is not None:
        sel_idx = _selected_channel_indices_from_jsonl(selected_channels, eeg_channel_jsonl)
        x = x[:, sel_idx, :]

    imgs = np.array(loaded["img"])
    if avg_trials:
        if imgs.ndim == 2:
            imgs = imgs[:, 0]
        imgs = imgs.reshape(-1)[: x.shape[0]]
    else:
        imgs = imgs.reshape(-1)

    image_ids = [Path(p).stem for p in imgs.tolist()]
    if len(image_ids) != x.shape[0]:
        raise ValueError(
            f"EEG/image mismatch: {x.shape[0]} vs {len(image_ids)} for {pt_path}"
        )

    x_np = x.float().cpu().numpy()  # [N, C, T]
    C, T = x_np.shape[1], x_np.shape[2]

    features = datasets.Features(
        {
            "eeg": datasets.Array2D(shape=(C, T), dtype="float32"),
            "image_id": datasets.Value("string"),
        }
    )

    ds = datasets.Dataset.from_dict(
        {
            "eeg": list(x_np),
            "image_id": image_ids,
        },
        features=features,
    )
    return ds


In [3]:
import torch
import torch.nn as nn

class ATM(nn.Module):
    def __init__(self, num_channels: int = 63, time_len: int = 250, embed_dim: int = 1024,
                 nhead: int = 8, num_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.num_channels = num_channels
        self.time_len = time_len
        self.embed_dim = embed_dim

        # 1. 初始投影：将 63 个通道的特征投影到 embed_dim
        # 论文逻辑：每一个时间点 $T$ 对应的 63 个电极信号被看作一个整体特征
        self.input_proj = nn.Linear(num_channels, embed_dim)

        # 2. 【关键】可学习的 [CLS] Token
        # 用于替代 mean()，它会自动学习如何从 250 个时间点中提取最重要的特征
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # 3. 位置编码：针对时间轴 T + [CLS] 的长度 (250 + 1)
        self.pos_embed = nn.Parameter(torch.zeros(1, time_len + 1, embed_dim))
        
        # 4. Transformer：处理时间序列
        # 注意：batch_first=True, 这样显存管理更高效
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=4*embed_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. MLP Projector (贴合论文，输出最终的 CLIP 对齐特征)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim)
        )

    def forward(self, eeg: torch.Tensor) -> torch.Tensor:
        # eeg shape: [B, 63, 250] (Batch, Channels, Time)
        B, C, T = eeg.shape 
        
        # --- 步骤 1: 调整维度以对齐时间序列 ---
        # [B, 63, 250] -> [B, 250, 63]
        x = eeg.permute(0, 2, 1) 
        
        # --- 步骤 2: 通道维度投影 ---
        # [B, 250, 63] -> [B, 250, 1024]
        x = self.input_proj(x)
        
        # --- 步骤 3: 拼接 [CLS] Token (不使用 Mean 的核心) ---
        # cls_tokens shape: [B, 1, 1024]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        # 拼接后 shape: [B, 251, 1024] (Seq_Len 变成了 251)
        x = torch.cat((cls_tokens, x), dim=1)
        
        # --- 步骤 4: 加上位置编码 ---
        x = x + self.pos_embed
        
        # --- 步骤 5: 经过 Transformer ---
        # 这里 Batch=128, Seq=251, 显存占用极低
        x = self.transformer(x) # [B, 251, 1024]
        
        # --- 步骤 6: 提取 [CLS] 对应的输出 ---
        # 我们只取第 0 个位置的向量，它已经通过 Attention 融合了 250 个时刻的信息
        res = x[:, 0, :] # [B, 1024]
        
        # --- 步骤 7: 最终映射 ---
        return self.mlp(res)

In [4]:
@torch.no_grad()
def evaluate_retrieval(model, test_loader, device="cuda"):
    # 处理 DataParallel 包装的模型
    model_to_eval = model.module if isinstance(model, nn.DataParallel) else model
    model_to_eval.eval()
    
    all_eeg_embeds = []
    all_clip_embeds = []
    
    try:
        for batch_idx, (eeg, clip_feat) in enumerate(test_loader):
            eeg = eeg.to(device)
            clip_feat = clip_feat.to(device)
            
            # 获取预测并归一化
            out = F.normalize(model(eeg), dim=-1)
            all_eeg_embeds.append(out.cpu())

            # 统一 CLIP 特征和模型输出的精度
            clip_feat = clip_feat.to(out.dtype)

            # 对真值也做归一化
            all_clip_embeds.append(F.normalize(clip_feat, dim=-1).cpu())
    except Exception as e:
        print(f"Error during evaluation: {e}")
        raise

    print(f"EEG embeds count: {len(all_eeg_embeds)}")
    print(f"CLIP embeds count: {len(all_clip_embeds)}")
    
    eeg_embeds = torch.cat(all_eeg_embeds, dim=0)
    clip_embeds = torch.cat(all_clip_embeds, dim=0)
    
    print(f"EEG embeddings shape: {eeg_embeds.shape}")
    print(f"CLIP embeddings shape: {clip_embeds.shape}")
    
    # 计算全局相似度矩阵
    similarity = torch.matmul(eeg_embeds, clip_embeds.T)
    labels = torch.arange(len(eeg_embeds))
    
    top1, top5 = 0, 0
    _, topk_indices = similarity.topk(k=min(5, len(eeg_embeds)), dim=1)
    
    for i in range(len(labels)):
        if labels[i] == topk_indices[i, 0]:
            top1 += 1
        if labels[i] in topk_indices[i]:
            top5 += 1
            
    acc1 = (top1 / len(labels)) * 100
    acc5 = (top5 / len(labels)) * 100
    print(f"\n--- 评估结果 ---")
    print(f"Top-1 Accuracy: {acc1:.2f}%")
    print(f"Top-5 Accuracy: {acc5:.2f}%")
    return acc1, acc5

In [5]:
def precompute_clip_for_split(data_root: Path, split: str = "train", 
                              model_name: str = "ViT-L/14@336px", 
                              batch_size: int = 128, 
                              save_path: Optional[str] = None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pt_path = data_root / f"{split}.pt"
    data = torch.load(pt_path, weights_only=False, map_location=device)
    print(f"Loaded {split}.pt → keys: {list(data.keys())}")
    
    raw_img = data['img']
    img_names = []

    # ======================================
    # 🔥 终极修复：解析字符串格式的列表
    # ======================================
    for item in raw_img:
        try:
            # 把 "['a.jpg','a.jpg']" 转成真正的列表
            parsed = ast.literal_eval(str(item))
            name = str(parsed[0]).strip()
        except:
            name = str(item).strip()
        
        img_names.append(name)

    # 图片根目录
    image_root = data_root / f"{split}_images" / f"{split}_images"
    print(f"✅ 图片根目录: {image_root.absolute()}")

    # 建立文件名映射
    name_map = {}
    for img_path in image_root.rglob("*.[jpJP][pnPN][gG]*"):
        name_map[img_path.name] = img_path

    print(f"✅ 找到 {len(name_map)} 张图片")

    # 模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model, preprocess = clip.load(model_name, device=device)
    clip_model.eval()
    # 转换为float32以避免精度不匹配
    clip_model = clip_model.float()
    
    # 【关键】使用hook获取投影层前的1024维特征
    visual_features = {}
    def hook_fn(module, input, output):
        # ln_post后的输出（投影前的1024维）
        # output形状: [batch_size, 1024]
        visual_features['feat'] = output
    
    hook = clip_model.visual.ln_post.register_forward_hook(hook_fn)
    
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(img_names), batch_size):
            batch_names = img_names[i:i+batch_size]
            batch_imgs = []
            
            for path_str in batch_names:
                # 取出纯文件名 aardvark_01b.jpg
                filename = Path(path_str).name
                
                if filename in name_map:
                    img = Image.open(name_map[filename]).convert("RGB")
                    batch_imgs.append(preprocess(img))
                else:
                    print(f"⚠️ 缺失文件: {filename}")
                    batch_imgs.append(torch.zeros(3,224,224))
            
            if batch_imgs:
                batch_tensor = torch.stack(batch_imgs).to(device)
                # 触发forward过程，hook会捕获ln_post的输出
                _ = clip_model.visual(batch_tensor)
                # 获取hook捕获的1024维特征
                emb = visual_features['feat']
                embeddings.append(emb.cpu())
            
            # 进度
            processed = i + len(batch_names)
            if processed % 500 == 0 or processed == len(img_names):
                print(f"已处理 {processed}/{len(img_names)}")
    
    # 移除hook
    hook.remove()
    
    all_emb = torch.cat(embeddings, dim=0)
    save_path = save_path or str(data_root / f"{split}_clip.pt")
    torch.save(all_emb, save_path)
    print(f"\n🎉 成功！保存到: {save_path} | 形状: {all_emb.shape}")
    
    return all_emb

In [6]:
# ====================== 2. CLIP Loss（温度改为论文的 0.07） ======================
def clip_loss(eeg_emb, img_emb, temp=0.07):
    eeg_emb = eeg_emb.float()
    img_emb = img_emb.float()
    
    B = eeg_emb.shape[0]
    img_emb = img_emb.view(B, -1)   # 确保是 (B, D)

    eeg_emb = F.normalize(eeg_emb, dim=-1)
    img_emb = F.normalize(img_emb, dim=-1)
    
    logits = eeg_emb @ img_emb.T / temp
    labels = torch.arange(B, device=eeg_emb.device)
    loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
    return loss

def contrastive_loss(eeg_embeds, clip_embeds, temperature=0.07):
    # 【核心修复】确保两者都是 float32 精度
    eeg_embeds = eeg_embeds.float()
    clip_embeds = clip_embeds.float()
    
    eeg_embeds = F.normalize(eeg_embeds, dim=-1)
    clip_embeds = F.normalize(clip_embeds, dim=-1)
    
    # 此时两者都是 float32，不会再报 dtype 错误
    logits = torch.matmul(eeg_embeds, clip_embeds.T) / temperature
    
    batch_size = eeg_embeds.shape[0]
    labels = torch.arange(batch_size, device=eeg_embeds.device)
    
    loss_eeg = F.cross_entropy(logits, labels)
    loss_clip = F.cross_entropy(logits.T, labels)
    return (loss_eeg + loss_clip) / 2

In [7]:
class EEGProjectDataset(Dataset):
    """Dataset class using load_eeg_dataset from EEG Project Sample Code."""
    
    def __init__(self, data_directory: Union[str, Path], split: str = "train", 
                 clip_pt_path: Optional[str] = None, map_location="cpu"):
        """
        Load EEG dataset using the standard load_eeg_dataset function.
        
        Parameters
        ----------
        data_directory : str or Path
            Directory containing train.pt, test.pt, and EEG_CHANNELS.jsonl
        split : str
            Either 'train' or 'test'
        clip_pt_path : str, optional
            Path to precomputed CLIP embeddings
        map_location : str
            Device for loading (default: 'cpu')
        """
        # Load EEG dataset using standard function
        dataset = load_eeg_dataset(
            data_directory=data_directory,
            split=split,
            avg_trials=True,
            selected_channels=None,
            eeg_channel_jsonl=str(Path(data_directory) / "EEG_CHANNELS.jsonl"),
        )
        
        # Convert to tensor format
        eeg_list = []
        for sample in dataset:
            eeg_tensor = torch.from_numpy(np.array(sample['eeg'])).float()
            eeg_list.append(eeg_tensor)
        
        self.eeg = torch.stack(eeg_list)  # [N, C, T]
        
        # Load CLIP features
        if clip_pt_path:
            self.clip = torch.load(clip_pt_path, map_location=map_location, weights_only=False)
            if isinstance(self.clip, np.ndarray):
                self.clip = torch.from_numpy(self.clip).float()
        else:
            raise ValueError("clip_pt_path must be provided")
        
        # Ensure matching sizes
        if len(self.eeg) != len(self.clip):
            print(f"⚠️ Warning: EEG size ({len(self.eeg)}) != CLIP size ({len(self.clip)})")
            min_size = min(len(self.eeg), len(self.clip))
            self.eeg = self.eeg[:min_size]
            self.clip = self.clip[:min_size]
            print(f"✅ Truncated to {min_size} samples")
        
        print(f"--- EEG 数据加载完成 ---")
        print(f"EEG 形状: {self.eeg.shape}")
        print(f"CLIP 形状: {self.clip.shape}")

    def __getitem__(self, index):
        return self.eeg[index], self.clip[index]

    def __len__(self):
        return len(self.eeg)

In [8]:
def train_atm(model, train_loader, epochs=40, lr=3e-4, device="cuda"):
    model.to(device)
    
    # 多卡训练支持
    if isinstance(device, str) and device.startswith('cuda'):
        device_ids = list(range(torch.cuda.device_count()))
        if len(device_ids) > 1:
            print(f"🚀 使用 DataParallel 进行多卡训练，GPU数量: {len(device_ids)}")
            model = nn.DataParallel(model, device_ids=device_ids)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    model.train()
    print(f"开始训练，设备: {device}, Epochs: {epochs}")
    
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (eeg, clip_feat) in enumerate(train_loader):
            eeg, clip_feat = eeg.to(device), clip_feat.to(device)
            
            optimizer.zero_grad()
            outputs = model(eeg)
            
            # 使用对比损失
            loss = contrastive_loss(outputs, clip_feat)
            
            loss.backward()
            # 梯度裁剪防止爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            running_loss += loss.item()
            
        scheduler.step()
        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
            
    return model

In [9]:
# ====================== 5. Retrieval (200-way) ======================
@torch.no_grad()
def retrieval(model: nn.Module, test_eeg: torch.Tensor, test_clip_db: torch.Tensor) -> Tuple[float, float]:
    # 处理 DataParallel 包装的模型
    model_to_eval = model.module if isinstance(model, nn.DataParallel) else model
    model_to_eval.eval()
    
    device = next(model.parameters()).device
    test_eeg = test_eeg.to(device)
    eeg_emb = F.normalize(model(test_eeg), dim=-1)
    db_emb = F.normalize(test_clip_db.to(device), dim=-1)

    sim = eeg_emb @ db_emb.T
    top1 = (sim.argmax(dim=1) == torch.arange(len(sim), device=device)).float().mean().item() * 100
    _, top5_idx = sim.topk(5, dim=1)
    top5 = (top5_idx == torch.arange(len(sim), device=device).unsqueeze(1)).any(dim=1).float().mean().item() * 100
    return top1, top5

In [10]:
# ====================== 6. 简单重建器 (Li et al. two-stage placeholder) ======================
class SimpleReconstructor(nn.Module):
    def __init__(self, embed_dim: int = 512):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, 512 * 8 * 8),
            nn.Unflatten(1, (512, 8, 8)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh()
        )

    def forward(self, eeg_emb: torch.Tensor) -> torch.Tensor:
        return self.decoder(eeg_emb)  # 输出 (B, 3, 64, 64) → 可上采样后用于SSIM/CLIP Score

In [11]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_root = Path("./jhaidata/image-eeg-data")
    
    # 检测GPU数量
    num_gpus = torch.cuda.device_count()
    print(f"🖥️ 可用GPU数量: {num_gpus}")
    
    # ====================== 第一步：预计算 CLIP embeddings（如果还没有的话）======================
    train_clip_path = str(data_root / "train_clip.pt")
    test_clip_path = str(data_root / "test_clip.pt")
    
    if not Path(train_clip_path).exists():
        print("🚀 正在预计算训练集 CLIP embedding...")
        precompute_clip_for_split(data_root, split="train", model_name="ViT-L/14@336px", save_path=train_clip_path)
    else:
        print(f"✅ 训练集 CLIP embedding 已存在: {train_clip_path}")
    
    if not Path(test_clip_path).exists():
        print("🚀 正在预计算测试集 CLIP embedding...")
        precompute_clip_for_split(data_root, split="test", model_name="ViT-L/14@336px", save_path=test_clip_path)
    else:
        print(f"✅ 测试集 CLIP embedding 已存在: {test_clip_path}")
    
    # ====================== 第二步：加载数据集 ======================
    train_dataset = EEGProjectDataset(str(data_root), split="train", clip_pt_path=train_clip_path)
    test_dataset = EEGProjectDataset(str(data_root), split="test", clip_pt_path=test_clip_path)
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Test dataset size: {len(test_dataset)}")
    
    # ====================== 第三步：创建 DataLoader（根据GPU数量调整batch_size）======================
    
    batch_size = 512
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size,
        shuffle=True, 
        num_workers=4 if num_gpus > 1 else 0,
        pin_memory=True,
        drop_last=False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size,
        shuffle=False, 
        num_workers=4 if num_gpus > 1 else 0,
        pin_memory=True,
        drop_last=False
    )
    
    print(f"✅ Batch size: {batch_size}")
    print(f"✅ Num workers: {4 if num_gpus > 1 else 0}")

    # ====================== 第四步：训练模型 ======================
    model = ATM().to(device)
    
    print("\n" + "="*60)
    print("开始训练 ATM 模型")
    print("="*60)
    trained_model = train_atm(model, train_loader, epochs=40, lr=3e-4, device=device)
    # 论文中 epochs=40, lr=3e-4


🖥️ 可用GPU数量: 1
🚀 正在预计算训练集 CLIP embedding...
Loaded train.pt → keys: ['eeg', 'label', 'img', 'text', 'session', 'ch_names', 'times']
✅ 图片根目录: /hpc2hdd/home/dsaa2012_009/jhaidata/image-eeg-data/train_images/train_images
✅ 找到 16540 张图片
已处理 16000/16540
已处理 16540/16540

🎉 成功！保存到: jhaidata/image-eeg-data/train_clip.pt | 形状: torch.Size([16540, 1024])
✅ 测试集 CLIP embedding 已存在: jhaidata/image-eeg-data/test_clip.pt
--- EEG 数据加载完成 ---
EEG 形状: torch.Size([16540, 63, 250])
CLIP 形状: torch.Size([16540, 1024])
--- EEG 数据加载完成 ---
EEG 形状: torch.Size([200, 63, 250])
CLIP 形状: torch.Size([200, 1024])
Train dataset size: 16540
Test dataset size: 200
✅ Batch size: 128
✅ Num workers: 0

开始训练 ATM 模型
开始训练，设备: cuda, Epochs: 5
Epoch [5/5], Loss: 4.4849, LR: 0.000000


In [12]:
# 执行评估
evaluate_retrieval(trained_model, test_loader, device=device)

EEG embeds count: 2
CLIP embeds count: 2
EEG embeddings shape: torch.Size([200, 1024])
CLIP embeddings shape: torch.Size([200, 1024])

--- 评估结果 ---
Top-1 Accuracy: 1.50%
Top-5 Accuracy: 7.50%


(1.5, 7.5)

In [13]:
# 执行评估
# evaluate_retrieval(trained_model, test_loader, device=device)

In [14]:
# 重建还没搞